# // Initialise Imports //

The cell below imports all required libraries for the analysis.

In [1]:
################################
### // Initilaise Imports // ###
################################

from image_to_model import *

--- [DEBUG] Starting Imports in image_proc_funcs.py ---
--- [DEBUG] Importing numpy...
--- [DEBUG] Importing skimage...
--- [DEBUG] Importing tqdm...
--- [DEBUG] Importing h5py...
--- [DEBUG] Importing scipy...
--- [DEBUG] Importing pandas...
--- [DEBUG] Importing ast...
--- [DEBUG] Importing standard libs (copy, random, json, re, subprocess, glob)...
--- [DEBUG] Importing pathlib...
--- [DEBUG] ALL IMPORTS SUCCESSFUL ---


# // Initialise File Paths //

The cell below initialises all required file paths for the anaylsis.

In [2]:
#########################################################################
### // Initialise this_dir, resources, CA_root, and src file paths // ###
#########################################################################

this_dir = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().parent
CA_root = Path(__file__).resolve().parent.parent.parent if "__file__" in globals() else Path.cwd().parent
resources_path = CA_root / Path("resources")

src_path = os.path.join(CA_root, "src")
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
#################################################
### // Initialise target image files paths // ###
#################################################

image_path_0 = CA_root / Path("dale_experimental/resources/batch_process_output_folder/image_0_seg.h5") # Image 0
image_path_1 = CA_root / Path("dale_experimental/resources/batch_process_output_folder/image_1_seg.h5") # Image 1
image_path_2 = CA_root / Path("dale_experimental/resources/batch_process_output_folder/image_2_seg.h5") # Image 2
image_path_3 = CA_root / Path("dale_experimental/resources/batch_process_output_folder/image_3_WKY_CB.h5") # Image 2
image_path_list = np.array([image_path_0, image_path_1, image_path_2, image_path_3])

################################################
### // Initialise file paths for pipeline // ###
################################################

ilastik_path = Path("/home/dsas627/Desktop/ilastik-1.4.1rc2-gpu-Linux/run_ilastik.sh") ### TODO: Delete/uncomment after docker integration is working
model_path = CA_root / Path("dale_experimental/resources/segmentation_model.ilp")
input_batch_processing_path = CA_root / Path("dale_experimental/resources/batch_process_input_folder")
output_batch_processing_path = CA_root / Path("dale_experimental/resources/batch_process_output_folder")

######################################
### // Verify Path(s) Existence // ###
######################################

# create a dictionary of variable_name : path_value
paths_to_check = {
    "this_dir": this_dir,
    "CA_root": CA_root,
    "resources_path": resources_path,
    "src_path": src_path,
    "image_path_0": image_path_0,
    "image_path_1": image_path_1,
    "image_path_2": image_path_2,
    "ilastik_path": ilastik_path,
    "model_path": model_path,
    "input_batch_processing_path": input_batch_processing_path,
    "output_batch_processing_path": output_batch_processing_path
}

missing_paths = []

for var_name, path_obj in paths_to_check.items():
    # cast to string to handle both pathlib.Path objects and standard strings
    if not os.path.exists(str(path_obj)):
        missing_paths.append(f"{var_name}: {path_obj}")

if not missing_paths:
    print("All Paths Set Successfully!")
else:
    print(f"Error: Non-existent file paths detected ({len(missing_paths)}):")
    for error in missing_paths:
        print(f" - {error}")

All Paths Set Successfully!


# // Run Image-To-Model Pipeline //

The cell below runs the image-to-model pipeline.

In [3]:
#####################################
### // Initialise image target // ###
#####################################

### TODO: User to edit image_selection_index depending on which image (Image 0, Image 1, or Image 2) they would like to process.
image_selection_index = 3 ### Change value to 0, 1, or 2 depending on which image you would like to input into the pipeline
target_image_path = image_path_list[image_selection_index]

#############################################################
### // Specify sub-volume (<= 0.1) of image to process // ###
#############################################################

### TODO: User to specify desired sub-volume. Ensure the value is <= 0.1 to avoid long processing times.
sub_volume = 0.1

##########################
### // Run pipeline // ###
##########################

run_ilastik_batch_processing = False ### TODO: Change to True if you want to use in-python Ilastik image segmentation

run_image_to_model(target_image_path, resources_path, ilastik_path, model_path,
                   input_batch_processing_path, output_batch_processing_path, sub_volume, run_ilastik_batch_processing)

print("Wrote:", str(CA_root / Path("dale_experimental/resources/user_output/image_to_model_vessel_array.csv")))
print("Wrote:", str(CA_root / Path("dale_experimental/resources/user_output/image_to_model_parameters.csv")))

### Note: Change label to 1!!!!

Successfully loaded data: (435, 357, 351), uint8
DEBUG: Unique values in loaded data: [0 1]

Processing sub-volume. Original shape: (435, 357, 351)
  Cropped segmentation to sub-volume. New shape: (43, 35, 35)
  Sub-volume ZYX voxel start offset from original: (196, 161, 158)
Using voxel spacing (X,Y,Z): (0.62517, 0.62517, 0.9030483)

  Processing label 1...
    Label 1 surface for seg-only plot prepared.
    Generating 3D skeleton for label 1...
      [Fix] Smoothing size is 1, skipping smoothing step.
      Skeletonizing...
    Skeleton: 363 voxels found.
    Added centerline for label 1.
        Calculating distance transform for radius estimation for label 1...
        Distance transform calculated.
      Detecting nodes (junctions/endpoints) using ndimage for label 1...
        Found 18 raw junction centroids.
        Found 13 raw endpoint centroids.
        Applying heuristics to define inlets/outlets...
        Node types updated with inlet/outlet heuristics.
        Added 18 'j

      Filtering for the largest connected component...
        Identified giant component with 23 nodes. Removing 8 nodes from small, disconnected fragments.
         Discretizing network with grid (and propagating voxel paths)...
        Calculating flow direction for the filtered and discretized graph...

      Classifying and saving final network data...
      Purging 6 edges with near-zero length or radius...
         Performing strong topological pruning (removing floating islands)...
          Pruned 39 disconnected edges (floating islands).
         Verifying the filtered adjacency matrix...
           Verification successful: No all-zero rows and columns found. ✅
        Edge adjacency matrix saved.
        Edge list saved.
          Segment connectivity data saved.

      Generating new visual actors from filtered network data (Voxel-First Method)...
        Mapping 31 original voxel segments to the filtered graph...


         Snapping intersection nodes to skeleton for visual accuracy...
Generating Vessel Array...
Assigning Network Boundary Conditions...
Network Solved!


Generating Parameter Array...
DONE :: Parameters array file for model image_to_model generated and saved.
Populating Parameter Array...
Wrote: /home/dsas627/PycharmProjects/circulatory_autogen/dale_experimental/resources/user_output/image_to_model_vessel_array.csv
Wrote: /home/dsas627/PycharmProjects/circulatory_autogen/dale_experimental/resources/user_output/image_to_model_parameters.csv
